# Course 18 — Brownfield Development with Agentic Coding Agents

Use the Northstar `AI-2410` vertical slice to discover current behavior, classify it, preserve semantic compatibility, and test a reversible migration. The notebook imports the same reusable application code and evaluation runner used by CI.

In [ ]:
import json
import sys
from pathlib import Path
for module_name in list(sys.modules):
    if module_name == 'lab':
        del sys.modules[module_name]
sys.path.insert(0, str(Path.cwd()))
from lab import build_report, load_json
from brownfield_app.contracts import ExecutionContext, Submission
from brownfield_app.discovery import validate_discovery_manifest
from brownfield_app.governance import EXPECTED_REPOSITORY_REVISION, EXPECTED_SPECIFICATION_REVISION, check_v1_semantics, select_summary_provider
from brownfield_app.legacy import AISummaryProvider
from brownfield_app.migration import ProposalStore

## 1. Establish the bounded reference result

Passing fixtures support integration-review eligibility only. They do not grant merge, deployment, retirement, or production authority.

In [ ]:
report = build_report()
assert report['reference']['state'] == 'READY_FOR_INTEGRATION_REVIEW'
assert report['reference']['merge_authorized'] is False
assert report['reference']['deployment_authorized'] is False
assert report['reference']['legacy_retirement_authorized'] is False
{key: report[key] for key in ('target_equation', 'baseline_comparison', 'reference')}

## 2. Invalidate stale discovery

A detailed manifest is still unusable when it describes a different repository revision.

In [ ]:
manifest = load_json('fixtures/discovery-manifest.json')
assert validate_discovery_manifest(manifest, actual_repository_revision=EXPECTED_REPOSITORY_REVISION).state == 'PASS'
stale = dict(manifest, repository_revision='previous-commit')
stale_decision = validate_discovery_manifest(stale, actual_repository_revision=EXPECTED_REPOSITORY_REVISION)
assert stale_decision.state == 'BLOCKED'
[finding.code for finding in stale_decision.findings]

## 3. Catch a semantic compatibility break

The response shape can remain valid while the HIGH-risk review invariant changes.

In [ ]:
submission = Submission('SUB-2410-001', 91, ('DOC-11@4',))
unsafe_v1 = {'submission_id': submission.submission_id, 'risk_category': 'HIGH', 'review_required': False}
semantic_decision = check_v1_semantics(submission, unsafe_v1)
assert semantic_decision.state == 'BLOCKED'
[finding.code for finding in semantic_decision.findings]

## 4. Separate a feature flag from authorization

Exposure is enabled, but the actor lacks the `create_summary_proposal` capability. Execution must stop before provider invocation.

In [ ]:
context = ExecutionContext(EXPECTED_REPOSITORY_REVISION, EXPECTED_SPECIFICATION_REVISION, 'observer-3', frozenset({submission.submission_id}), frozenset({'ai_risk_summary'}), frozenset())
provider, decision = select_summary_provider(context, authorized=True)
assert provider is None
assert decision.state == 'BLOCKED'
[finding.code for finding in decision.findings]

## 5. Exercise idempotency at the state boundary

The unique state transition—not a caller-side pre-check—prevents a duplicate proposal.

In [ ]:
proposal = AISummaryProvider().generate(submission)
store = ProposalStore()
assert store.migrate(proposal, 'legacy-summary@7') is True
assert store.migrate(proposal, 'legacy-summary@7') is False
assert store.count() == 1
assert store.authoritative_count() == 0
{'records': store.count(), 'authoritative_records': store.authoritative_count()}

## 6. Interpret the evidence honestly

The deterministic suite contains twelve golden cases, twenty-four labelled mutations, and ten focused experiments. This establishes reproducibility for the teaching fixture. Production work still needs real repository/runtime evidence, representative model evaluation, enterprise identity, database-specific migration testing, rollout monitoring, and accountable authorization.